**Sabrina Urriola**
**Scenario**  
You've been hired by a startup company called HomeGuard to develop a prototype for their smart home monitoring system. The system uses various sensors throughout a house to detect potential security issues, comfort problems, and equipment failures. Your job is to create a simulator that processes sensor readings and triggers appropriate alerts.

### Step 1: Setting Up the Project

Do: Create your project file and set up the basic structure.

Do:
1. Create a new Python file named `homeguard_system.py` (or `homeguard_system.ipynb` if you prefer a notebook)
2. Add a header comment with your name and the project description
3. Import the necessary modules (`random` for generating sensor readings and `datetime` for timestamps)

Verify: You should have a Python file that runs without errors (though it won't do anything yet).

**Checkpoint:** Run your file (`python homeguard_system.py` for a `.py` file, or "Run All" in your notebook). It should execute without errors.

In [13]:
import random as rd
import datetime as dt


### Step 2: Define the sensors and the alerts (dictionaries)

**A sensor** is one dictionary with 4 fields:

| Field | Example |
|---|---|
| `id` | `2` |
| `location` | `"Front Door"` |
| `type` | `"door"` (or `"motion"`, `"temperature"`, `"smoke"`) |
| `current_value` | `None` at the start, because nothing has been read yet |

Write a function that creates the 4 sensors of the house and puts them in a **list**.

**Check:** print your sensor list. You see 4 dictionaries, each with `current_value: None`.


In [14]:
def create_sensors():
    return [
        {"id": 1, "location": "Living Room", "type": "temperature", "current_value": None},
        {"id": 2, "location": "Living Room", "type": "motion", "current_value": None},
        {"id": 3, "location": "Living Room", "type": "smoke", "current_value": None},
        {"id": 4, "location": "Front Door", "type": "door", "current_value": None},
    ]

sensors = create_sensors()
print(sensors)

[{'id': 1, 'location': 'Living Room', 'type': 'temperature', 'current_value': None}, {'id': 2, 'location': 'Living Room', 'type': 'motion', 'current_value': None}, {'id': 3, 'location': 'Living Room', 'type': 'smoke', 'current_value': None}, {'id': 4, 'location': 'Front Door', 'type': 'door', 'current_value': None}]


The one key rule: event + mode

Every reading is an **event**: the door opened, smoke appeared, the kitchen got cold.

The house also has a **mode**, which tells the program who is home:
- **HOME**: the family is inside
- **AWAY**: nobody is home
- **SLEEP**: the family is asleep *(optional)*



Your program combines the two:

```
EVENT  +  MODE  =  ALERT or NOT
```

The same event can be normal or dangerous, depending on the mode:

| Event | HOME | AWAY | SLEEP *(optional)* |
|---|---|---|---|
| Motion or door opens | nothing | 🚨 HIGH security | ⚠️ MEDIUM security |
| Smoke | 🔥 CRITICAL | 🔥 CRITICAL | 🔥 CRITICAL |
| Temp below 35°F or above 95°F | ⚠️ safety alert | ⚠️ safety alert | ⚠️ safety alert |
| Temp outside 65–75°F | 💡 comfort notice | nothing | nothing |

> *The sensor tells you what happened. The mode tells you if it matters.*
**An alert** is also a dictionary. Its fields come from the sample output:

```
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
            ────  ────────  ─────────────────────────────────────
         severity category                 message
```

| Field | Example | Meaning |
|---|---|---|
| `time` | `14:31:00` | When it happened (from the `[LOG]` line) |
| `severity` | `HIGH` | How serious it is: CRITICAL, HIGH, MEDIUM or LOW |
| `category` | `SECURITY` | What kind of problem: SAFETY, SECURITY or COMFORT |
| `message` | `Front Door opened while in AWAY mode!` | What exactly happened |

Also store the mode in a variable: `system_mode = "AWAY"`.

### Step 3: Write the conditions (if/else)
- Save the limits as constants at the top, for example `FREEZE_TEMP = 35`.
- Write **one function** that takes a sensor and the mode, and uses if/else to follow the rule table from section 2.
- It **returns an alert** if something is wrong, or **`None`** if everything is fine.

**Check:** set a value by hand and test it. For example, a door that is open in AWAY mode returns an alert, and the same door in HOME mode returns `None`.

In [15]:
FREEZE_TEMP = 35
MAX_SAFE_TEMP = 95
COMFORT_MIN_TEMP = 65
COMFORT_MAX_TEMP = 75


def check_sensor(sensor, mode):
    sensor_type = sensor["type"]
    value = sensor["current_value"]
    location = sensor["location"]
    mode = mode.upper()

    if value is None:
        return None

    severity = category = message = None

    if sensor_type in ("door", "motion"):
        is_active = value is True or str(value).lower() in ("open", "opened", "motion", "true")
        if is_active and mode in ("AWAY", "SLEEP"):
            severity = "HIGH" if mode == "AWAY" else "MEDIUM"
            category = "SECURITY"
            message = f"{location} {'opened' if sensor_type == 'door' else 'detected motion'} while in {mode} mode!"
    elif sensor_type == "smoke" and bool(value):
        severity = "CRITICAL"
        category = "SAFETY"
        message = f"Smoke detected in {location}!"
    elif sensor_type == "temperature":
        if value < FREEZE_TEMP or value > MAX_SAFE_TEMP:
            severity = "HIGH"
            category = "SAFETY"
            message = f"Unsafe temperature in {location}: {value}°F!"
        elif mode == "HOME" and not COMFORT_MIN_TEMP <= value <= COMFORT_MAX_TEMP:
            severity = "LOW"
            category = "COMFORT"
            message = f"Uncomfortable temperature in {location}: {value}°F."

    if message is None:
        return None

    return {
        "time": dt.datetime.now().strftime("%H:%M:%S"),
        "severity": severity,
        "category": category,
        "message": message,
    }

### Step 4: Building Functions

Do: Create functions to generate sensor readings, process them, trigger alerts, and log events.

Do:
1. Write a function to generate realistic sensor readings
2. Create a function to process readings and determine if alerts are needed
3. Implement a function to trigger and display alerts
4. Add a logging function to record all events

**Explanation:** Functions are reusable blocks of code that perform specific tasks. They help organize your code and make it easier to maintain.

Verify: You have all the core functions needed to process sensor data and trigger alerts.

In [16]:
system_mode = "AWAY"
event_log = []


def generate_sensor_reading(sensor):
    """Generate and store a simulated reading for one sensor."""
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        reading = rd.choice(temperature_sensor)
    elif sensor_type == "motion":
        reading = rd.random() < 0.15
    elif sensor_type == "door":
        reading = rd.random() < 0.05
    elif sensor_type == "smoke":
        reading = rd.random() < 0.02
    else:
        raise ValueError(f"Unknown sensor type: {sensor_type}")

    sensor["current_value"] = reading
    return reading


def log_event(sensor):
    """Record and print a sensor reading."""
    entry = {
        "time": dt.datetime.now().strftime("%H:%M:%S"),
        "sensor_id": sensor["id"],
        "location": sensor["location"],
        "type": sensor["type"],
        "reading": sensor["current_value"],
    }
    event_log.append(entry)
    print(
        f"[LOG] {entry['time']} — {entry['location']} "
        f"{entry['type']} reading: {entry['reading']}"
    )


def trigger_alert(alert):
    """Display an alert."""
    print(
        f"[ALERT!] 🚨 {alert['severity']}: {alert['category']}: "
        f"{alert['message']}"
    )


def process_readings(sensor_list, mode):
    """Generate readings, log each event, and display any alerts."""
    alerts = []

    for sensor in sensor_list:
        generate_sensor_reading(sensor)
        log_event(sensor)

        alert = check_sensor(sensor, mode)
        if alert is not None:
            trigger_alert(alert)
            alerts.append(alert)

    return alerts


alerts = process_readings(sensors, system_mode)

[LOG] 17:26:57 — Living Room temperature reading: 32
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: 32°F!
[LOG] 17:26:57 — Living Room motion reading: False
[LOG] 17:26:57 — Living Room smoke reading: False
[LOG] 17:26:57 — Front Door door reading: False


### Step 5: Creating Classes

Do: Convert the sensor dictionary into a Sensor class with methods.

Do:
1. Define a Sensor class with an `__init__` method
2. Add methods: `read()`, `isAbnormal()`, and `reset()`
3. Update your code to use the Sensor class instead of dictionaries

**Explanation:** Classes are blueprints for creating objects. They combine data (properties) and functions (methods) into a single unit. This makes your code more organized and easier to work with.

Verify: You have a Sensor class and can create sensor objects.

**Checkpoint:** Test your class:

In [17]:
class Sensor:
    def __init__(self, id, location, type, current_value=None):
        self.id = id
        self.location = location
        self.type = type
        self.current_value = current_value

    def read(self):
        """Generate and store a simulated reading."""
        if self.type == "temperature":
            self.current_value = rd.choice(temperature_sensor)
        elif self.type == "motion":
            self.current_value = rd.random() < 0.15
        elif self.type == "door":
            self.current_value = rd.random() < 0.05
        elif self.type == "smoke":
            self.current_value = rd.random() < 0.02
        else:
            raise ValueError(f"Unknown sensor type: {self.type}")

        return self.current_value

    def isAbnormal(self, mode="AWAY"):
        """Return an alert if this reading is abnormal; otherwise return None."""
        return check_sensor(
            {
                "id": self.id,
                "location": self.location,
                "type": self.type,
                "current_value": self.current_value,
            },
            mode,
        )

    def reset(self):
        """Clear the current reading."""
        self.current_value = None


# Convert the existing sensor dictionaries to Sensor objects.
sensors = [Sensor(**sensor) for sensor in sensors]

# Test the class methods.
for sensor in sensors:
    reading = sensor.read()
    print(f"{sensor.location} {sensor.type}: {reading}")
    print("Alert:", sensor.isAbnormal(system_mode))
    sensor.reset()

Living Room temperature: 3
Alert: {'time': '17:26:57', 'severity': 'HIGH', 'category': 'SAFETY', 'message': 'Unsafe temperature in Living Room: 3°F!'}
Living Room motion: False
Alert: None
Living Room smoke: False
Alert: None
Front Door door: False
Alert: None


### Step 6: Integrating Everything

Do: Combine all components into a complete simulator that runs continuously.

Do:
1. Create a main simulation loop
2. Integrate all your functions and classes
3. Format the output to match the sample

Verify: You have a complete, working simulator that runs and displays formatted output.

**Checkpoint:** Run your complete program and verify it produces output similar to the sample.

In [23]:
import time

def run_simulation(max_cycles=None, interval=2):
    """Run the HomeGuard simulator for a set number of cycles or continuously."""
    cycle = 0

    try:
        while max_cycles is None or cycle < max_cycles:
            cycle += 1
            print(f"\n--- Simulation cycle {cycle} | Mode: {system_mode} ---")

            for sensor in sensors:
                sensor.read()

                # log_event expects a dictionary, so adapt the Sensor object.
                sensor_data = {
                    "id": sensor.id,
                    "location": sensor.location,
                    "type": sensor.type,
                    "current_value": sensor.current_value,
                }
                log_event(sensor_data)

                alert = sensor.isAbnormal(system_mode)
                if alert is not None:
                    trigger_alert(alert)
                    alerts.append(alert)

            if max_cycles is None or cycle < max_cycles:
                time.sleep(interval)

    except KeyboardInterrupt:
        print("\nSimulation stopped.")


# Run 3 cycles for a notebook-friendly test.
run_simulation(max_cycles=3, interval=2)

# To run continuously instead, use:
# run_simulation(max_cycles=None, interval=2)


--- Simulation cycle 1 | Mode: AWAY ---
[LOG] 17:30:16 — Living Room temperature reading: 32
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: 32°F!
[LOG] 17:30:16 — Living Room motion reading: False
[LOG] 17:30:16 — Living Room smoke reading: False
[LOG] 17:30:16 — Front Door door reading: False

--- Simulation cycle 2 | Mode: AWAY ---
[LOG] 17:30:18 — Living Room temperature reading: 32
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: 32°F!
[LOG] 17:30:18 — Living Room motion reading: False
[LOG] 17:30:18 — Living Room smoke reading: False
[LOG] 17:30:18 — Front Door door reading: False

--- Simulation cycle 3 | Mode: AWAY ---
[LOG] 17:30:20 — Living Room temperature reading: 8
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: 8°F!
[LOG] 17:30:20 — Living Room motion reading: False
[LOG] 17:30:20 — Living Room smoke reading: False
[LOG] 17:30:20 — Front Door door reading: True
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!


### Step 7: Testing Your System

Do: Verify your system works correctly and handles all scenarios.

Do:
1. Run your simulator multiple times to see different scenarios
2. Test different system modes (HOME, AWAY, SLEEP)
3. Verify that alerts trigger correctly
4. Check that the output is formatted properly

**Test scenarios:**
1. **Security Test:** Run with `system_mode="AWAY"` and verify security alerts trigger
2. **Safety Test:** Check that temperature and smoke alerts work
3. **Comfort Test:** Run with `system_mode="HOME"` and verify comfort notifications appear

**Observable output format:**
```
=== HomeGuard Security System ===
Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
```

**Checkpoint:** Your simulator should run without errors and display formatted output for all sensor types.

In [25]:
original_mode = system_mode

print("=== HomeGuard Security System: Mode Tests ===")
print(f"Time: {dt.datetime.now().strftime('%H:%M:%S')}")

try:
    for mode in ("AWAY", "HOME", "SLEEP"):
        system_mode = mode
        print(f"\nTesting {mode} mode:")
        run_simulation(max_cycles=1, interval=0)
finally:
    system_mode = original_mode

print("\nMode tests complete.")

=== HomeGuard Security System: Mode Tests ===
Time: 17:36:21

Testing AWAY mode:

--- Simulation cycle 1 | Mode: AWAY ---
[LOG] 17:36:21 — Living Room temperature reading: -6
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: -6°F!
[LOG] 17:36:21 — Living Room motion reading: False
[LOG] 17:36:21 — Living Room smoke reading: False
[LOG] 17:36:21 — Front Door door reading: False

Testing HOME mode:

--- Simulation cycle 1 | Mode: HOME ---
[LOG] 17:36:21 — Living Room temperature reading: 20
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: 20°F!
[LOG] 17:36:21 — Living Room motion reading: False
[LOG] 17:36:21 — Living Room smoke reading: False
[LOG] 17:36:21 — Front Door door reading: False

Testing SLEEP mode:

--- Simulation cycle 1 | Mode: SLEEP ---
[LOG] 17:36:21 — Living Room temperature reading: -9
[ALERT!] 🚨 HIGH: SAFETY: Unsafe temperature in Living Room: -9°F!
[LOG] 17:36:21 — Living Room motion reading: False
[LOG] 17:36:21 — Living Room smoke reading: Fals